# Attention-Augmented memTrace — single combined experiment
All 167 Reduced features plus all specified attention families. One combined random forest; run smoke first, then one full evaluation. This is a StarCoder2 code adaptation, not exact original memTrace.


In [ ]:
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
UPLOAD_SOURCE = ""
UPLOAD_ZIP = ""
PROJECT_ROOT = WORK_ROOT / "attention-memtrace-source-v1"
BASELINE_RUN = "/kaggle/input/datasets/sayonsujitmondal/sersem-baseline-1000-results3"
PATH_OVERRIDES = dict(shadow="", validation="", calibration="", heldout="", scores="", config="")
ADAPTIVE_SCORES = ""
REDUCED_MEMTRACE_SCORES = ""  # optional exact heldout_memtrace_scores.parquet; provenance-checked sibling features may be reused
MODE = "smoke"
SMOKE_SIZES = dict(shadow=80, validation=20, calibration=20, heldout=30)
RUN_ID = f"attention-memtrace-{MODE}-v1"
OUT = WORK_ROOT / "attention_memtrace_outputs" / RUN_ID
RESUME_FROM = ""
CACHE_REUSE_FROM = ""  # full: completed combined smoke feature_cache.sqlite; resume: prior full cache via RESUME_FROM
MODEL_CACHE_DIR = ""
MODEL_LOCAL_FILES_ONLY = True
ALLOW_MODEL_DOWNLOAD = False
DATASET_CACHE_DIR = WORK_ROOT / "hf_cache/datasets"
SOURCE_ACCESS = "streaming"
SESSION_BUDGET_MINUTES = 690
DISK_RESERVE_GIB = 3.0
WORK_ROOT.mkdir(parents=True, exist_ok=True)


## Verify source package, then initialize the existing pinned environment
No model libraries are imported before installation. Source discovery accepts ZIP or extracted package.


In [ ]:
import os, subprocess, sys
SOURCE_MANIFEST_SHA256 = 'd955d68afb7a9eb91811fb8790d3e3754bc4c6c40fea8b7cdede1dbc90a1e4bc'
"""Stdlib-only verified source discovery/extraction, callable before dependency setup."""
import hashlib
import json
from pathlib import Path
import shutil
import zipfile

MARKER = 'attention_memtrace_code_attack/source_manifest.json'
ZIP_NAME = 'attention_memtrace_kaggle_source.zip'


def stage_source(input_root, project_root, expected_sha, upload_source='', upload_zip=''):
    root, project = Path(input_root), Path(project_root)
    markers = sorted(root.rglob('source_manifest.json'))
    candidates = sorted({p.parent.parent for p in markers if p.parent.name=='attention_memtrace_code_attack'} |
                        {p for p in root.rglob(ZIP_NAME) if not (p.parent/MARKER).is_file()})
    print('Attention memTrace source candidates:', [str(p) for p in candidates])
    if not (project/MARKER).is_file():
        source = Path(upload_zip) if upload_zip else Path(upload_source) if upload_source else None
        if source is None:
            if len(candidates) != 1:
                raise FileNotFoundError('Attach attention_memtrace_kaggle_source.zip or extracted package; set UPLOAD_SOURCE/UPLOAD_ZIP if ambiguous. Candidates: '+str(candidates))
            source = candidates[0]
        if source.is_dir() and not (source/MARKER).is_file():
            source = source/ZIP_NAME
        if project.exists():
            raise FileExistsError('Incomplete PROJECT_ROOT; select a fresh directory')
        def safe(name):
            if Path(name).is_absolute() or '..' in Path(name).parts:
                raise ValueError('Unsafe source path: '+name)
            destination = (project/name).resolve()
            if not destination.is_relative_to(project.resolve()):
                raise ValueError('Unsafe source path: '+name)
            return destination
        if source.is_dir():
            raw = (source/MARKER).read_bytes()
            if hashlib.sha256(raw).hexdigest() != expected_sha:
                raise ValueError('Outdated source manifest')
            manifest = json.loads(raw)
            for name in list(manifest)+[MARKER]:
                destination = safe(name)
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source/name, destination)
        else:
            with zipfile.ZipFile(source) as archive:
                if hashlib.sha256(archive.read(MARKER)).hexdigest() != expected_sha:
                    raise ValueError('Outdated ZIP manifest')
                if len(archive.namelist()) != len(set(archive.namelist())):
                    raise ValueError('Duplicate ZIP entries')
                for entry in archive.infolist():
                    safe(entry.filename)
                    if entry.external_attr >> 16 & 0o170000 == 0o120000:
                        raise ValueError('Source ZIP symlinks forbidden')
                archive.extractall(project)
    raw = (project/MARKER).read_bytes()
    if hashlib.sha256(raw).hexdigest() != expected_sha:
        raise ValueError('Source manifest pin mismatch')
    manifest = json.loads(raw)
    for name, expected in manifest.items():
        path = (project/name).resolve()
        if not path.is_relative_to(project.resolve()) or hashlib.sha256(path.read_bytes()).hexdigest()!=expected:
            raise ValueError('Source hash mismatch: '+name)
    return manifest

source_manifest = stage_source(INPUT_ROOT, PROJECT_ROOT, SOURCE_MANIFEST_SHA256, UPLOAD_SOURCE, UPLOAD_ZIP)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "SERSEM-Poisoned-Chalice-Competition-2026/code"))


In [ ]:
# Set process policy before importing any experiment or model libraries.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# The system dictionary is needed by the pinned pyenchant dependency.
packages = ["libenchant-2-2", "hunspell-en-us"]
status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *packages], capture_output=True, text=True)
if status.returncode != 0 or status.stdout.splitlines().count("install ok installed") != len(packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *packages], check=True)
from shared.kaggle_setup import install_dependencies, validate_environment
install_report = install_dependencies(PROJECT_ROOT / "shared/requirements-kaggle.txt", WORK_ROOT)
# A correct fresh-kernel rerun is a no-op. If setup changed loaded modules,
# install_dependencies raises an explicit one-restart instruction.
environment = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt", require_cuda=True)


# Exercise the required CUDA runtime and cuBLAS shared libraries through actual APIs.
import torch
_cuda_check = torch.ones((2,2), device="cuda", dtype=torch.float16)
assert torch.isfinite(_cuda_check @ _cuda_check).all()
torch.cuda.synchronize()
del _cuda_check
environment["cuda_shared_library_api_check"] = "CUDA allocation and FP16 cuBLAS matmul passed"


In [ ]:
# CPU tensors and tiny randomly initialized StarCoder2; no pretrained downloads.
for test_file in ('test_attention_memtrace.py', 'test_kaggle_setup.py'):
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                    '-p', test_file, '-v'], cwd=PROJECT_ROOT, check=True)


## Load and validate every original saved partition
Smoke selection is deterministic with seed 42; the scientific configuration is identical in full mode.


In [ ]:
import copy, random
import numpy as np
import pandas as pd
from adaptive_token_attack.data import resolve_paths,load_run,select_partitions,frame_fingerprint
from adaptive_token_attack.features import load_frozen_target
from attention_memtrace_code_attack.cache import Cache,fingerprint
from attention_memtrace_code_attack.data import scoring_identity,prepare_plans,fixed_configuration,verified_reduced_prefixes
from attention_memtrace_code_attack.inference import Extractor
from attention_memtrace_code_attack.runner import run_features
from attention_memtrace_code_attack.evaluation import exact_match,evaluate
from attention_memtrace_code_attack import METHOD_NAME,METHOD_STATUS
from shared.resource_pipeline import Monitor,restore_run,file_hash
from shared.resource_data import SourceStore
from shared.sersem_baseline import write_json
fixed = fixed_configuration()
print(METHOD_NAME, METHOD_STATUS)
paths = resolve_paths(BASELINE_RUN,PATH_OVERRIDES,INPUT_ROOT)
original,saved_scores,base_cfg = load_run(paths)
if SMOKE_SIZES != fixed['smoke_counts']:
    raise ValueError('Smoke counts are fixed to 80/20/20/30')
partitions,selection = select_partitions(original,MODE,SMOKE_SIZES,seed=fixed['seed'])
if set(pd.concat(list(original.values())).language) != {'Go','Java','Python','Ruby','Rust'}:
    raise ValueError('Unexpected saved programming languages')
if base_cfg['versions'] != environment['versions']:
    raise ValueError('Restore the saved baseline numerical/API environment before inference/comparison')
# Optional comparisons cannot block extraction. Missing, ambiguous or invalid files are omitted.
optional_scores, optional_hashes = {}, {}
reduced_scores_path = None
optional_specs = {
    'adaptive_score': ('heldout_adaptive_scores.parquet', ADAPTIVE_SCORES, 'adaptive_score'),
    'reduced_memtrace_score': ('heldout_memtrace_scores.parquet', REDUCED_MEMTRACE_SCORES, 'memtrace_score'),
}
for column, (filename, explicit, original_column) in optional_specs.items():
    candidates = [Path(explicit)] if explicit else sorted(INPUT_ROOT.rglob(filename))
    if len(candidates) != 1:
        print('Optional comparison unavailable/ambiguous:', column, candidates)
        continue
    try:
        candidate = pd.read_parquet(candidates[0]).rename(columns={original_column:column})
        exact_match(partitions['heldout'], candidate, [column])
        optional_scores[column] = candidate
        optional_hashes[column] = file_hash(candidates[0])
        if column == 'reduced_memtrace_score':
            reduced_scores_path = candidates[0]
    except Exception as exc:
        print('Optional comparison omitted:', column, str(exc))
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {'.','..'}:
    raise ValueError('RUN_ID must be a directory name')
if OUT.resolve().is_relative_to(INPUT_ROOT.resolve()):
    raise ValueError('OUT must be writable and separate from inputs')
if (OUT.resolve().is_relative_to(paths['config'].parent.resolve())
        or any(p.resolve().is_relative_to(OUT.resolve()) for p in paths.values())):
    raise ValueError('Outputs must be separate from baseline inputs')
OUT.mkdir(parents=True,exist_ok=True)
if RESUME_FROM:
    if json.loads((Path(RESUME_FROM)/'config.json').read_text()).get('mode') != MODE:
        raise ValueError('Resume must use the same mode; use CACHE_REUSE_FROM for smoke to full')
    restore_run(RESUME_FROM,OUT)
source_cfg = copy.deepcopy(base_cfg)
source_cfg['resources'].update(data_access=SOURCE_ACCESS,text_cache_gib=.1,
    session_budget_minutes=SESSION_BUDGET_MINUTES,disk_reserve_gib=DISK_RESERVE_GIB)
monitor = Monitor(source_cfg,OUT)
monitor.guard()
print('Selected:',{k:len(v) for k,v in partitions.items()})
if SOURCE_ACCESS == 'cached':
    os.environ['HF_DATASETS_CACHE'] = str(DATASET_CACHE_DIR)


## Freeze the complete combined schema and provenance
1787 features: unchanged 167-feature prefix followed by all 1620 attention features. A full run requires compatible completed smoke evidence; same-mode full resume preserves that evidence.


In [ ]:
from shared.cache_setup import ensure_model_snapshot
from transformers import AutoConfig,AutoTokenizer
cache_dir = MODEL_CACHE_DIR or base_cfg['resources']['model_cache_dir']
ensure_model_snapshot(base_cfg['model_id'],base_cfg['model_revision'],cache_dir,
    ALLOW_MODEL_DOWNLOAD,DISK_RESERVE_GIB,WORK_ROOT,OUT/'model_cache_setup.json')
options = dict(revision=base_cfg['model_revision'],cache_dir=str(cache_dir),local_files_only=MODEL_LOCAL_FILES_ONLY)
tokenizer = AutoTokenizer.from_pretrained(base_cfg['model_id'],use_fast=True,**options)
model_config = AutoConfig.from_pretrained(base_cfg['model_id'],**options)
identity = scoring_identity(base_cfg,tokenizer,model_config,source_manifest)
reduced_prefixes, prefix_reuse_report = verified_reduced_prefixes(reduced_scores_path, partitions, identity)
# Exclude location/error text from mathematical cache identity. Verified content hashes remain bound.
identity['prefix_reuse'] = {k:v for k,v in prefix_reuse_report.items() if k != 'reason'}
write_json(OUT/'reduced_prefix_reuse.json',prefix_reuse_report)
if MODE == 'full':
    from attention_memtrace_code_attack.workflow import require_completed_smoke
    if not CACHE_REUSE_FROM and not (OUT/'smoke_validation.json').is_file():
        smoke_cache = WORK_ROOT/'attention_memtrace_outputs/attention-memtrace-smoke-v1/feature_cache.sqlite'
        if smoke_cache.is_file():
            CACHE_REUSE_FROM = str(smoke_cache)
    require_completed_smoke(identity,CACHE_REUSE_FROM,OUT)
experiment = dict(schema='combined-attention-memtrace-experiment-v1',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    paper_faithful=False,mode=MODE,selection=selection,scoring=identity,
    source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    inputs={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in paths.items()},
    optional_scores_sha256=optional_hashes)
if (OUT/'config.json').exists() and json.loads((OUT/'config.json').read_text()) != experiment:
    raise ValueError('Execution identity changed; choose a new RUN_ID')
write_json(OUT/'config.json',experiment)
write_json(OUT/'environment.json',environment)
write_json(OUT/'reproducibility.json',fixed)
write_json(OUT/'research_access.json',json.loads((PROJECT_ROOT/'attention_memtrace_code_attack/research_access.json').read_text()))
write_json(OUT/'feature_schema.json',identity['schema'])
write_json(OUT/'attention_feature_schema.json',identity['schema']['attention'])
write_json(OUT/'source_version_hashes.json',dict(source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    sources=source_manifest,versions=environment,scoring_sha256=fingerprint(identity)))
write_json(OUT/'input_paths.json',dict(**{k:str(v) for k,v in paths.items()},optional_comparisons=optional_hashes))
for role,frame in partitions.items():
    frame.to_parquet(OUT/(role+'_manifest.parquet'),index=False)
write_json(OUT/'run_manifest.json',dict(status='preparing',method_name=METHOD_NAME,paper_faithful=False,
    mode=MODE,selection=selection,scoring_sha256=fingerprint(identity)))
cache = Cache(OUT/'feature_cache.sqlite',identity)
if CACHE_REUSE_FROM:
    monitor.guard(Path(CACHE_REUSE_FROM).stat().st_size)
    cache.reuse_from(CACHE_REUSE_FROM)
source_store = SourceStore(source_cfg,OUT,monitor)
plans = prepare_plans(partitions,tokenizer,identity,cache,source_store,monitor)
for sample_id, vector in reduced_prefixes.items():
    plans[sample_id]['verified_reduced_features'] = vector
print('Prepared plans:',len(plans),'Combined features:',len(identity['schema']['names']))

## Five real attention-enabled shadow probes, then extraction
Both allocated and reserved peaks must remain at most 11 GiB. Attention modules are timed using read-only CUDA-event hooks. Missing layers/heads or invalid probabilities stop extraction. No attention features are dropped.


In [ ]:
def extractor_factory():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Missing predictions require Kaggle GPU')
    monitor.guard()
    random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    # Supported eager load, same immutable weights/tokenizer, exactly one model copy.
    eager_cfg = copy.deepcopy(base_cfg)
    eager_cfg['resources']['attention_backend'] = 'eager'
    model,loaded_tokenizer = load_frozen_target(eager_cfg,cache_dir,MODEL_LOCAL_FILES_ONLY)
    if hashlib.sha256(loaded_tokenizer.backend_tokenizer.to_str().encode()).hexdigest() != identity['tokenizer_sha256']:
        raise ValueError('Inference tokenizer differs from prepared plans')
    if model.config.num_hidden_layers != identity['schema']['attention']['layers']:
        raise ValueError('Loaded architecture differs from frozen feature schema')
    return Extractor(model, identity['schema'])
write_json(OUT/'run_manifest.json',dict(status='scoring',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    paper_faithful=False,mode=MODE,selection=selection,scoring_sha256=fingerprint(identity),
    source_manifest_sha256=SOURCE_MANIFEST_SHA256))
try:
    features,timing = run_features(partitions,plans,identity,cache,extractor_factory,monitor,OUT)
except BaseException as exc:
    cache.close()
    write_json(OUT/'run_manifest.json',dict(status='interrupted',method_name=METHOD_NAME,
        paper_faithful=False,mode=MODE,reason=str(exc),scoring_sha256=fingerprint(identity)))
    raise
import gc,torch
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print(json.dumps(timing,indent=2))

## Fit the fixed random forest; calibrate and evaluate
Shadow trains the classifier. Validation is diagnostic only. Calibration selects the deployed threshold. Held-out ROC cutoffs remain diagnostic.


In [ ]:
report = evaluate(features,partitions,saved_scores,optional_scores,identity,OUT,MODE,timing)
monitor.guard()
cache.close()
write_json(OUT/'run_manifest.json',dict(status='complete',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    paper_faithful=False,mode=MODE,selection=selection,scoring_sha256=fingerprint(identity),
    source_manifest_sha256=SOURCE_MANIFEST_SHA256,input_hashes=experiment['inputs'],
    optional_scores_sha256=optional_hashes,
    selected_counts={k:len(v) for k,v in partitions.items()},coverage=report['coverage'],timing=timing,
    span_summary=dict(truncated_samples=int(features.truncated.sum()), maximum_source_tokens=int(features.source_token_count.max())),
    artifacts={p.name:file_hash(p) for p in OUT.iterdir()
        if p.is_file() and p.name in {'features.parquet','feature_schema.json','attention_feature_schema.json','resource_probe.json','classifier.joblib','config.json','metrics.json','heldout_attention_memtrace_scores.parquet','matched_comparison.parquet',
            'feature_cache.sqlite','source_version_hashes.json','threshold.json','timing.json','reproducibility.json','research_access.json'}}))
print(json.dumps(report,indent=2))

## Download and preserve results
Download the ZIP below and Save Version with outputs. For an interrupted extraction, download the partial results ZIP and use RESUME_FROM in a later session. Model weights and HF caches are excluded.


In [ ]:
from attention_memtrace_code_attack.archive import create_results_archive
from IPython.display import FileLink, display
archive = create_results_archive(OUT, WORK_ROOT, RUN_ID, DISK_RESERVE_GIB)
print('Results archive:', archive)
display(FileLink(str(archive)))


## Optional: preserve an interrupted run
Run this cell manually only after extraction stops. It closes the feature cache and archives partial artifacts.


In [ ]:
# Uncomment after a resource pause or interruption; run instead of the completed-results cell.
# cache.close()  # safe after extraction has stopped, including an already-closed connection
# from IPython.display import FileLink, display
# from attention_memtrace_code_attack.archive import create_partial_archive
# archive = create_partial_archive(OUT, WORK_ROOT, RUN_ID, DISK_RESERVE_GIB)
# display(FileLink(str(archive)))
